In [ ]:
#script para verificar a vriancia do dataset por origem 
import pandas as pd
import os
import numpy as np

In [ ]:
# Here I combine datasets with the same source (e.g., ce-*)
def combine_datasets(path, substring):
    files = [f for f in os.listdir(path) if substring in f and f.endswith('.csv')]
    
    if not files:
        print(f"No files containing the substring '{substring}' were found.")
        return None
    
    dataframe_list = []
    for file in files:
        file_path = os.path.join(path, file)
        df = pd.read_csv(file_path)
        dataframe_list.append(df)
    
    combined_df = pd.concat(dataframe_list, ignore_index=True)
    return combined_df

path = '../../datasets/serie-multivariada'
dataframes_by_source = {}

for dirs, root, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            source = '-' + file.split('-')[2] + '-'
            #print(f"Processing files with source: {source}")
            df = combine_datasets(path, source)
            
            if df is not None:
                key_name = f"{source.strip('-')}" 
                dataframes_by_source[key_name] = df


In [ ]:
#carrego meu dataset com os rmse de cada modelo por link de origem 
rmse_bbr =pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_bbr_rmse.csv')
rmse_cubic =pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_cubic_rmse.csv')

In [ ]:
# bbr - variancia 
correlations_by_model = {}

variances = {}
for key, value in dataframes_by_source.items():
    dataset = dataframes_by_source[key]
        
    variance = dataset[['Vazao_bbr']].var().iloc[0]
    variances[key] = variance

models = rmse_bbr.columns[1:]  

for model in models:
    df_model = pd.DataFrame({
        'Source': rmse_bbr['source'],
        'RMSE': rmse_bbr[model],
        'Variance': rmse_bbr['source'].map(variances)
    })
    correlation = df_model[['RMSE', 'Variance']].corr().iloc[0, 1]
    correlations_by_model[model] = correlation

print("Correlation between high RMSE and dataset variance for each model:")
for model, correlation in correlations_by_model.items():
    print(f"{model} - BBR: {correlation:.4f}")


In [ ]:
# cubic - variancia 
correlations_by_model = {}

variances = {}
for key, value in dataframes_by_source.items():
    dataset = dataframes_by_source[key]
        
    variance = dataset[['Vazao_cubic']].var().iloc[0]
    variances[key] = variance

models =rmse_cubic.columns[1:]  

for model in models:
    df_model = pd.DataFrame({
        'Source':rmse_cubic['source'],
        'RMSE':rmse_cubic[model],
        'Variance':rmse_cubic['source'].map(variances)
    })
    correlation = df_model[['RMSE', 'Variance']].corr().iloc[0, 1]
    correlations_by_model[model] = correlation

print("Correlation between high RMSE and dataset variance for each model:")
for model, correlation in correlations_by_model.items():
    print(f"{model} - CUBIC: {correlation:.4f}")


In [ ]:
#coeficiente de variancia

# bbr
correlations_by_model = {}

variances = {}
for key, value in dataframes_by_source.items():
    dataset = dataframes_by_source[key]
        
    variance_coef = (dataset['Vazao_bbr'].std()/ dataset[['Vazao_bbr']].mean())*100
    variances[key] = variance_coef

models = rmse_bbr.columns[1:]  

for model in models:
    df_model = pd.DataFrame({
        'Source': rmse_bbr['source'],
        'RMSE': rmse_bbr[model],
        'Variance_coef': rmse_bbr['source'].map(variances)
    })
    correlation = df_model[['RMSE', 'Variance_coef']].corr().iloc[0, 1]
    correlations_by_model[model] = correlation

print("Correlation between high RMSE and dataset variance for each model:")
for model, correlation in correlations_by_model.items():
    print(f"{model} - BBR: {correlation:.4f}")


In [ ]:
# Coeficiente de variância

# Inicializando os dicionários
correlations_by_model = {}
variances = {}

# Calculando o coeficiente de variância para cada fonte
for key, value in dataframes_by_source.items():
    dataset = dataframes_by_source[key]
    variance_coef = (dataset['Vazao_cubic'].std() / dataset['Vazao_cubic'].mean()) * 100
    variances[key] = variance_coef

# Obtendo os nomes dos modelos
models = rmse_cubic.columns[1:]

# Calculando as correlações
for model in models:
    df_model = pd.DataFrame({
        'Source': rmse_cubic['source'],
        'RMSE': rmse_cubic[model],
        'Variance_coef': rmse_cubic['source'].map(lambda x: variances[x])  # Corrigindo para obter apenas o valor
    })
    correlation = df_model[['RMSE', 'Variance_coef']].corr().iloc[0, 1]
    correlations_by_model[model] = correlation

# Exibindo as correlações
print("Correlation between high RMSE and dataset variance for each model:")
for model, correlation in correlations_by_model.items():
    print(f"{model} - CUBIC: {correlation:.4f}")



In [ ]:
#gerando arquivos csv com coeficiente de variancia e nrmse de todos os modelos e links 

def process_rmse_and_variances(rmse_file, dataframes_by_source, target):
    # Certifique-se de que rmse_file seja um DataFrame
    if isinstance(rmse_file, str):  # Se for string, carregue como CSV
        rmse_file = pd.read_csv(rmse_file)
    variances = {}
    for key, _ in dataframes_by_source.items():
        dataset = dataframes_by_source[key]
        variance_coef = (dataset[target].std() / dataset[target].mean()) * 100
        variances[key] = variance_coef

    models = rmse_file.columns[1:]
    final_df = pd.DataFrame({'Source': rmse_file['source']})
    final_df['Variance_coef'] = final_df['Source'].map(lambda x: variances.get(x, None))
    for model in models:
        final_df[f'RMSE_{model}'] = rmse_file[model]
    output_path = f'../../results/regression/coefVariation_nrmse/Coefvariation_nrmse_{target}.csv'
    final_df.to_csv(output_path, index=False)
    return final_df


rmse = pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_bbr_rmse.csv')
result = process_rmse_and_variances(rmse, dataframes_by_source, 'Vazao_bbr')

